# C2U in RAG: Compositional Concept Unlearning
### Full Pipeline on Kaggle GPU (TOFU, MNLI, Amazon, ArXiv)
Notebook này thực thi toàn trình thực nghiệm C2U theo bài báo gốc:
1. **Cell 1:** Kiểm tra GPU & Cài đặt thư viện
2. **Cell 2:** Tự động nạp Mã nguồn & Dữ liệu
3. **Cell 3:** Chạy Kiểm thử Sanity Check Dry-run trên GPU
4. **Cell 4:** Huấn luyện ACR & Đánh giá C2U Top-5 Retrieval
5. **Cell 5:** Trực quan hóa kết quả bằng biểu đồ Seaborn/Matplotlib
6. **Cell 6:** Tải file CSV & Ảnh biểu đồ về máy tính

In [ ]:
# CELL 1: Kiểm tra phần cứng GPU & Cài đặt dependencies
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[CẢNH BÁO] Chưa bật GPU! Hãy vào mục Settings ở góc phải và bật Accelerator là GPU T4 x2 hoặc P100.")

!pip install -q sentence-transformers tabulate seaborn matplotlib

In [ ]:
# CELL 2: Thiết lập Mã nguồn & Dữ liệu (Tự động phát hiện bất kể tên Dataset trên Kaggle)
import os, sys, shutil, zipfile

WORKING_DIR = "/kaggle/working"
INPUT_DIR = "/kaggle/input"

if WORKING_DIR not in sys.path:
    sys.path.insert(0, WORKING_DIR)
src_path = os.path.join(WORKING_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

found_source = False

# Trường hợp 1: Tìm thấy file zip trong /kaggle/input
for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.endswith(".zip") and ("bundle" in file.lower() or "c2u" in file.lower()):
            zip_full = os.path.join(root, file)
            print(f"[Setup] Đang giải nén gói: {zip_full} vào {WORKING_DIR}...")
            with zipfile.ZipFile(zip_full, "r") as zf:
                zf.extractall(WORKING_DIR)
            found_source = True
            break
    if found_source:
        break

# Trường hợp 2: Kaggle đã tự động giải nén thư mục khi mount dataset
if not found_source:
    for root, dirs, files in os.walk(INPUT_DIR):
        if "run_kaggle_pipeline.py" in files:
            print(f"[Setup] Tìm thấy mã nguồn tại: {root}. Đang sao chép vào {WORKING_DIR}...")
            for item in os.listdir(root):
                s = os.path.join(root, item)
                d = os.path.join(WORKING_DIR, item)
                if os.path.isdir(s):
                    shutil.copytree(s, d, dirs_exist_ok=True)
                else:
                    shutil.copy2(s, d)
            found_source = True
            break

if found_source:
    print("[Setup] Hoàn tất thiết lập! Danh sách file trong /kaggle/working:")
    for item in sorted(os.listdir(WORKING_DIR)):
        print(f"  - {item}")
else:
    print("[LỖI] Không tìm thấy dữ liệu/mã nguồn C2U! Vui lòng bấm Add Data ở góc phải để thêm dataset c2u-bundle.")

In [ ]:
# CELL 3: Chạy Kiểm thử Sanity Check Dry-run trên GPU
# Xác nhận 100% gradient của ACR, giải lstsq, và vùng tin cậy Trust-Region hoạt động trơn tru
!python /kaggle/working/test_dry_run.py

In [ ]:
# CELL 4: Chạy Pipeline Thực nghiệm C2U
# Lựa chọn DATASET: "tofu", "mnli", "amazon", "arxiv" hoặc "all"
# Lựa chọn MODEL_NAME: 
#   - sentence-transformers/all-MiniLM-L6-v2 (d=384, N=768)
#   - BAAI/bge-base-en-v1.5 (d=768, N=1536)
#   - intfloat/e5-base-v2 (d=768, N=1536)

DATASET = "tofu"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EPOCHS = 15
BATCH_SIZE = 128

!python /kaggle/working/run_kaggle_pipeline.py \
    --dataset {DATASET} \
    --model_name {MODEL_NAME} \
    --work_dir /kaggle/working \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE}

In [ ]:
# CELL 5: Trực quan hóa Kết quả Thực nghiệm bằng Biểu đồ
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

csv_path = "/kaggle/working/c2u_final_results.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("=== BẢNG KẾT QUẢ C2U RETRIEVAL (TOP-5) ===")
    display(df)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.set_theme(style="whitegrid")

    # 1. Target Hit@5 Drop
    sns.barplot(data=df[df["Method"] == "C2U"], x="Dataset", y="Target_Hit@5_Drop", hue="Model", ax=axes[0], palette="Blues_d")
    axes[0].set_title("Target Hit@5 Drop (Càng cao càng tốt)", fontsize=12, fontweight="bold")
    axes[0].set_ylabel("Mức giảm tỷ lệ Target (%)")

    # 2. Control Utility Drop
    sns.barplot(data=df[df["Method"] == "C2U"], x="Dataset", y="Control_Utility_Drop", hue="Model", ax=axes[1], palette="Greens_d")
    axes[1].axhline(0, color="red", linestyle="--", alpha=0.7)
    axes[1].set_title("Control Utility Drop (Mục tiêu <= 0)", fontsize=12, fontweight="bold")
    axes[1].set_ylabel("Mức suy giảm tiện ích Control")

    # 3. Target Exposure Change
    sns.barplot(data=df[df["Method"] == "C2U"], x="Dataset", y="TE_Change", hue="Model", ax=axes[2], palette="Oranges_d")
    axes[2].axhline(0, color="red", linestyle="--", alpha=0.7)
    axes[2].set_title("Target Exposure Change in Control (<= 0)", fontsize=12, fontweight="bold")
    axes[2].set_ylabel("Thay đổi độ phơi bày")

    plt.tight_layout()
    plt.savefig("/kaggle/working/c2u_evaluation_plot.png", dpi=300)
    plt.show()
    print("[Plot] Đã lưu biểu đồ chất lượng cao vào /kaggle/working/c2u_evaluation_plot.png")
else:
    print("Chưa tìm thấy file kết quả c2u_final_results.csv. Hãy chạy Cell 4 trước!")

In [ ]:
# CELL 6: Tải về Checkpoint và File Báo cáo Kết quả
import os
from IPython.display import FileLink, display

print("Click vào liên kết dưới đây để tải kết quả về máy tính:")
if os.path.exists("/kaggle/working/c2u_final_results.csv"):
    display(FileLink("c2u_final_results.csv"))
if os.path.exists("/kaggle/working/c2u_evaluation_plot.png"):
    display(FileLink("c2u_evaluation_plot.png"))